# 03. Обучение RuBERT NER baseline

Ноутбук собирает все компоненты из YAML, обучает RuBERT token classifier и сохраняет лучший checkpoint по validation strict entity-level micro-F1. Перед запуском выполните `02_preprocessing.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import runpy

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
EXPERIMENT_CONFIG = PROJECT_DIR / "configs" / "experiments" / "ner_baseline_v1.yaml"
MANIFEST = PROJECT_DIR / "rurebus_data" / "processed" / "manifest.csv"
BOOTSTRAP = PROJECT_DIR / "colab_bootstrap.py"

for required_path in (EXPERIMENT_CONFIG, MANIFEST, BOOTSTRAP):
    if not required_path.is_file():
        raise FileNotFoundError(f"Не найден {required_path}. Проверьте PROJECT_DIR и актуальность файлов проекта.")

bootstrap_project = runpy.run_path(str(BOOTSTRAP))["bootstrap_project"]
bootstrap_project(PROJECT_DIR)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Предупреждение: обучение на CPU будет очень медленным.")

In [ ]:
from rurebus_ie.training import train_ner_experiment

summary = train_ner_experiment(EXPERIMENT_CONFIG, project_root=PROJECT_DIR)
print(f"Лучшая эпоха: {summary.best_epoch}")
print(f"Validation strict micro-F1: {summary.best_validation_f1:.4f}")
print(f"Checkpoint: {summary.checkpoint_dir}")

In [ ]:
import pandas as pd

history = pd.DataFrame(summary.history)
display(history)
history.plot(x="epoch", y=["train_loss", "validation_loss"], grid=True);
history.plot(x="epoch", y=["validation_micro_f1", "validation_macro_f1"], ylim=(0, 1), grid=True);